In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

tf.keras.utils.set_random_seed(42)

In [ ]:
data_dir = Path(
    "../data/processed/neural_sdm"
)

model_replicates = {}

for rep in range(1, 6):

    path = (
        data_dir /
        f"sapajus_neural_sdm_rep{rep}.csv"
    )

    model_replicates[rep] = pd.read_csv(
        path
    )

print("Replicates loaded:", list(model_replicates))

In [ ]:
for rep, data in model_replicates.items():

    assert len(data) == 1512
    assert data["fold"].nunique() == 5
    assert data["fold"].isna().sum() == 0
    assert (data["class"] == 1).sum() == 252
    assert (data["class"] == 0).sum() == 1260

print(
    "✓ All Neural SDM datasets loaded correctly."
)

In [ ]:
predictors = [
    "BIO1",
    "BIO4",
    "BIO12",
    "BIO15"
]

rep = 1
test_fold = 1
val_fold = 2

data = model_replicates[rep].copy()

train = data[
    ~data["fold"].isin([test_fold, val_fold])
].copy()

val = data[
    data["fold"] == val_fold
].copy()

test = data[
    data["fold"] == test_fold
].copy()

print("TRAIN")
print(train["class"].value_counts().sort_index())

print("\nVALIDATION")
print(val["class"].value_counts().sort_index())

print("\nTEST")
print(test["class"].value_counts().sort_index())

print(
    "\nTotal:",
    len(train) + len(val) + len(test)
)

In [ ]:
def get_xy(df):

    X = df[predictors].to_numpy(
        dtype=np.float32
    )

    y = df["class"].to_numpy(
        dtype=np.float32
    )

    return X, y


X_train, y_train = get_xy(train)
X_val, y_val = get_xy(val)
X_test, y_test = get_xy(test)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
).astype(np.float32)

X_val_scaled = scaler.transform(
    X_val
).astype(np.float32)

X_test_scaled = scaler.transform(
    X_test
).astype(np.float32)

In [ ]:
scaling_audit = pd.DataFrame({
    "predictor": predictors,

    "train_mean":
        X_train_scaled.mean(axis=0),

    "train_sd":
        X_train_scaled.std(axis=0),

    "val_mean":
        X_val_scaled.mean(axis=0),

    "val_sd":
        X_val_scaled.std(axis=0),

    "test_mean":
        X_test_scaled.mean(axis=0),

    "test_sd":
        X_test_scaled.std(axis=0)
})

scaling_audit.round(3)

In [ ]:
classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train.astype(int)
)

class_weight = {
    0: float(weights[0]),
    1: float(weights[1])
}

print(class_weight)

In [ ]:
def build_neural_sdm(seed=42):

    tf.keras.utils.set_random_seed(seed)

    model = keras.Sequential([
        layers.Input(
            shape=(len(predictors),)
        ),

        layers.Dense(
            16,
            activation="relu",
            kernel_regularizer=
                regularizers.l2(0.001)
        ),

        layers.Dense(
            8,
            activation="relu",
            kernel_regularizer=
                regularizers.l2(0.001)
        ),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss="binary_crossentropy",

        metrics=[
            keras.metrics.AUC(
                name="roc_auc"
            ),

            keras.metrics.AUC(
                curve="PR",
                name="pr_auc"
            )
        ]
    )

    return model


model = build_neural_sdm()

model.summary()

In [ ]:
print("TRAIN")
print(train["class"].value_counts().sort_index())

print("\nVALIDATION")
print(val["class"].value_counts().sort_index())

print("\nTEST")
print(test["class"].value_counts().sort_index())

In [ ]:
model = build_neural_sdm(seed=42)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=30,
    restore_best_weights=True,
    min_delta=1e-4,
    verbose=1
)

history = model.fit(
    X_train_scaled,
    y_train,

    validation_data=(
        X_val_scaled,
        y_val
    ),

    epochs=300,
    batch_size=32,

    class_weight=class_weight,

    callbacks=[
        early_stopping
    ],

    verbose=1
)

In [ ]:
best_epoch = (
    np.argmin(
        history.history["val_loss"]
    ) + 1
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    round(
        min(history.history["val_loss"]),
        4
    )
)

In [ ]:
history_df = pd.DataFrame(
    history.history
)

history_df["epoch"] = (
    np.arange(len(history_df)) + 1
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["loss"],
    label="Train"
)

ax.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Validation"
)

ax.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Best epoch = {best_epoch}"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Binary cross-entropy loss")
ax.set_title(
    "Neural SDM — Replicate 1 / Test Fold 1"
)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["roc_auc"],
    label="Train ROC-AUC"
)

ax.plot(
    history_df["epoch"],
    history_df["val_roc_auc"],
    label="Validation ROC-AUC"
)

ax.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.7
)

ax.set_xlabel("Epoch")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0, 1)

ax.set_title(
    "ROC-AUC during training"
)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["pr_auc"],
    label="Train PR-AUC"
)

ax.plot(
    history_df["epoch"],
    history_df["val_pr_auc"],
    label="Validation PR-AUC"
)

ax.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.7
)

ax.set_xlabel("Epoch")
ax.set_ylabel("PR-AUC")
ax.set_ylim(0, 1)

ax.set_title(
    "PR-AUC during training"
)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
candidate_models = {
    "A_8": {
        "hidden_units": [8],
        "l2": 0.001
    },

    "B_16_8": {
        "hidden_units": [16, 8],
        "l2": 0.001
    },

    "C_16_8_L2": {
        "hidden_units": [16, 8],
        "l2": 0.01
    }
}

In [ ]:
def build_candidate_model(
    hidden_units,
    l2_strength,
    seed=42
):

    tf.keras.utils.set_random_seed(seed)

    model = keras.Sequential()

    model.add(
        layers.Input(
            shape=(len(predictors),)
        )
    )

    for units in hidden_units:

        model.add(
            layers.Dense(
                units,
                activation="relu",
                kernel_regularizer=
                    regularizers.l2(
                        l2_strength
                    )
            )
        )

    model.add(
        layers.Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss="binary_crossentropy",

        metrics=[
            keras.metrics.AUC(
                name="roc_auc"
            ),

            keras.metrics.AUC(
                curve="PR",
                name="pr_auc"
            )
        ]
    )

    return model

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

selection_data = model_replicates[1].copy()

internal_folds = [2, 3, 4, 5]

selection_results = []

In [ ]:
for model_name, config in candidate_models.items():

    print(f"\n===== {model_name} =====")

    for val_fold in internal_folds:

        # ---------------------------------
        # Spatial split
        # ---------------------------------

        train = selection_data[
            ~selection_data["fold"].isin(
                [1, val_fold]
            )
        ].copy()

        val = selection_data[
            selection_data["fold"] == val_fold
        ].copy()

        # ---------------------------------
        # X / y
        # ---------------------------------

        X_train = train[
            predictors
        ].to_numpy(dtype=np.float32)

        y_train = train[
            "class"
        ].to_numpy(dtype=np.float32)

        X_val = val[
            predictors
        ].to_numpy(dtype=np.float32)

        y_val = val[
            "class"
        ].to_numpy(dtype=np.float32)

        # ---------------------------------
        # Scaling: TRAIN ONLY
        # ---------------------------------

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(
            X_train
        ).astype(np.float32)

        X_val_scaled = scaler.transform(
            X_val
        ).astype(np.float32)

        # ---------------------------------
        # Class weights: TRAIN ONLY
        # ---------------------------------

        weights = compute_class_weight(
            class_weight="balanced",
            classes=np.array([0, 1]),
            y=y_train.astype(int)
        )

        class_weight_inner = {
            0: float(weights[0]),
            1: float(weights[1])
        }

        # ---------------------------------
        # New model
        # ---------------------------------

        model_inner = build_candidate_model(
            hidden_units=
                config["hidden_units"],

            l2_strength=
                config["l2"],

            seed=42
        )

        early_stopping = (
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=30,
                min_delta=1e-4,
                restore_best_weights=True,
                verbose=0
            )
        )

        history_inner = model_inner.fit(
            X_train_scaled,
            y_train,

            validation_data=(
                X_val_scaled,
                y_val
            ),

            epochs=300,
            batch_size=32,

            class_weight=
                class_weight_inner,

            callbacks=[
                early_stopping
            ],

            verbose=0
        )

        # ---------------------------------
        # Validation predictions
        # ---------------------------------

        y_val_prob = (
            model_inner.predict(
                X_val_scaled,
                verbose=0
            )
            .ravel()
        )

        roc_auc = roc_auc_score(
            y_val,
            y_val_prob
        )

        pr_auc = average_precision_score(
            y_val,
            y_val_prob
        )

        best_epoch = (
            np.argmin(
                history_inner.history[
                    "val_loss"
                ]
            ) + 1
        )

        best_val_loss = min(
            history_inner.history[
                "val_loss"
            ]
        )

        selection_results.append({
            "model": model_name,
            "val_fold": val_fold,

            "n_train":
                len(train),

            "n_val":
                len(val),

            "n_train_presence":
                int(y_train.sum()),

            "n_val_presence":
                int(y_val.sum()),

            "best_epoch":
                best_epoch,

            "val_loss":
                best_val_loss,

            "roc_auc":
                roc_auc,

            "pr_auc":
                pr_auc
        })

        print(
            f"Fold {val_fold} | "
            f"ROC={roc_auc:.3f} | "
            f"PR={pr_auc:.3f} | "
            f"epoch={best_epoch}"
        )

In [ ]:
selection_results = pd.DataFrame(
    selection_results
)

selection_results.round(3)

In [ ]:
model_selection_summary = (
    selection_results
    .groupby("model")
    .agg(
        mean_roc_auc=("roc_auc", "mean"),
        sd_roc_auc=("roc_auc", "std"),

        mean_pr_auc=("pr_auc", "mean"),
        sd_pr_auc=("pr_auc", "std"),

        mean_val_loss=("val_loss", "mean"),
        sd_val_loss=("val_loss", "std"),

        median_best_epoch=(
            "best_epoch",
            "median"
        )
    )
    .sort_values(
        "mean_roc_auc",
        ascending=False
    )
)

model_selection_summary.round(3)

In [ ]:
data = model_replicates[1].copy()

development = data[
    data["fold"] != 1
].copy()

external_test = data[
    data["fold"] == 1
].copy()

print("DEVELOPMENT")
print(
    development["class"]
    .value_counts()
    .sort_index()
)

print("\nEXTERNAL TEST")
print(
    external_test["class"]
    .value_counts()
    .sort_index()
)

In [ ]:
X_dev = development[
    predictors
].to_numpy(dtype=np.float32)

y_dev = development[
    "class"
].to_numpy(dtype=np.float32)

X_test = external_test[
    predictors
].to_numpy(dtype=np.float32)

y_test = external_test[
    "class"
].to_numpy(dtype=np.float32)

final_scaler = StandardScaler()

X_dev_scaled = (
    final_scaler
    .fit_transform(X_dev)
    .astype(np.float32)
)

X_test_scaled = (
    final_scaler
    .transform(X_test)
    .astype(np.float32)
)

In [ ]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_dev.astype(int)
)

final_class_weight = {
    0: float(weights[0]),
    1: float(weights[1])
}

final_class_weight

In [ ]:
final_model = build_candidate_model(
    hidden_units=[16, 8],
    l2_strength=0.001,
    seed=42
)

final_history = final_model.fit(
    X_dev_scaled,
    y_dev,

    epochs=7,
    batch_size=32,

    class_weight=final_class_weight,

    verbose=1
)

In [ ]:
y_test_prob = (
    final_model.predict(
        X_test_scaled,
        verbose=0
    )
    .ravel()
)

external_roc_auc = roc_auc_score(
    y_test,
    y_test_prob
)

external_pr_auc = average_precision_score(
    y_test,
    y_test_prob
)

print(
    "External spatial ROC-AUC:",
    round(external_roc_auc, 3)
)

print(
    "External spatial PR-AUC:",
    round(external_pr_auc, 3)
)

In [ ]:
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve
)

# ROC
fpr, tpr, roc_thresholds = roc_curve(
    y_test,
    y_test_prob
)

# Precision-recall
precision, recall, pr_thresholds = (
    precision_recall_curve(
        y_test,
        y_test_prob
    )
)

In [ ]:
def evaluate_candidate(
    data,
    predictors,
    outer_test_fold,
    inner_val_fold,
    hidden_units,
    l2_strength,
    seed=42
):

    train = data[
        ~data["fold"].isin(
            [outer_test_fold, inner_val_fold]
        )
    ].copy()

    val = data[
        data["fold"] == inner_val_fold
    ].copy()

    X_train = train[predictors].to_numpy(
        dtype=np.float32
    )

    y_train = train["class"].to_numpy(
        dtype=np.float32
    )

    X_val = val[predictors].to_numpy(
        dtype=np.float32
    )

    y_val = val["class"].to_numpy(
        dtype=np.float32
    )

    # Fit scaler only on inner training data
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    ).astype(np.float32)

    X_val_scaled = scaler.transform(
        X_val
    ).astype(np.float32)

    # Class weights only from inner training
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=y_train.astype(int)
    )

    class_weight = {
        0: float(weights[0]),
        1: float(weights[1])
    }

    model = build_candidate_model(
        hidden_units=hidden_units,
        l2_strength=l2_strength,
        seed=seed
    )

    callback = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=30,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(
            X_val_scaled,
            y_val
        ),
        epochs=300,
        batch_size=32,
        class_weight=class_weight,
        callbacks=[callback],
        verbose=0
    )

    prob = model.predict(
        X_val_scaled,
        verbose=0
    ).ravel()

    best_epoch = (
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    return {
        "roc_auc": roc_auc_score(
            y_val,
            prob
        ),
        "pr_auc": average_precision_score(
            y_val,
            prob
        ),
        "val_loss": min(
            history.history["val_loss"]
        ),
        "best_epoch": best_epoch
    }

In [ ]:
candidate_models = {
    "A_8": {
        "hidden_units": [8],
        "l2": 0.001
    },

    "B_16_8": {
        "hidden_units": [16, 8],
        "l2": 0.001
    },

    "C_16_8_L2": {
        "hidden_units": [16, 8],
        "l2": 0.01
    }
}

data = model_replicates[1].copy()

nested_selection = []

In [ ]:
for outer_fold in range(1, 6):

    print(
        f"\n========== OUTER FOLD {outer_fold} =========="
    )

    inner_folds = [
        f for f in range(1, 6)
        if f != outer_fold
    ]

    for model_name, config in candidate_models.items():

        inner_scores = []

        for inner_val_fold in inner_folds:

            result = evaluate_candidate(
                data=data,
                predictors=predictors,
                outer_test_fold=outer_fold,
                inner_val_fold=inner_val_fold,
                hidden_units=config["hidden_units"],
                l2_strength=config["l2"],
                seed=42
            )

            result.update({
                "outer_fold": outer_fold,
                "model": model_name,
                "inner_val_fold":
                    inner_val_fold
            })

            nested_selection.append(
                result
            )

            inner_scores.append(
                result["roc_auc"]
            )

        print(
            model_name,
            "| mean inner ROC:",
            round(
                np.mean(inner_scores),
                3
            )
        )

In [ ]:
nested_selection = pd.DataFrame(
    nested_selection
)

nested_summary = (
    nested_selection
    .groupby(
        ["outer_fold", "model"]
    )
    .agg(
        mean_roc_auc=(
            "roc_auc",
            "mean"
        ),

        sd_roc_auc=(
            "roc_auc",
            "std"
        ),

        mean_pr_auc=(
            "pr_auc",
            "mean"
        ),

        mean_val_loss=(
            "val_loss",
            "mean"
        ),

        median_epoch=(
            "best_epoch",
            "median"
        )
    )
    .reset_index()
)

nested_summary.round(3)

In [ ]:
selected_models = (
    nested_summary
    .sort_values(
        [
            "outer_fold",
            "mean_roc_auc",
            "mean_pr_auc"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .groupby(
        "outer_fold",
        as_index=False
    )
    .first()
)

selected_models

In [ ]:
nested_selection.to_csv(
    "../data/processed/neural_sdm/"
    "nested_selection_rep1.csv",
    index=False
)

nested_summary.to_csv(
    "../data/processed/neural_sdm/"
    "nested_summary_rep1.csv",
    index=False
)

selected_models.to_csv(
    "../data/processed/neural_sdm/"
    "selected_models_rep1.csv",
    index=False
)

print("✓ Nested CV checkpoint saved.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

results_dir = Path(
    "../data/processed/neural_sdm"
)

nested_selection = pd.read_csv(
    results_dir / "nested_selection_rep1.csv"
)

nested_summary = pd.read_csv(
    results_dir / "nested_summary_rep1.csv"
)

selected_models = pd.read_csv(
    results_dir / "selected_models_rep1.csv"
)

print("Nested selection:", nested_selection.shape)
print("Nested summary:", nested_summary.shape)
print("Selected models:", selected_models.shape)

display(selected_models)

In [ ]:
assert len(selected_models) == 5

assert set(
    selected_models["outer_fold"]
) == {1, 2, 3, 4, 5}

assert selected_models[
    "model"
].eq("B_16_8").all()

assert selected_models[
    "median_epoch"
].notna().all()

print("✓ Nested CV recuperada corretamente.")
print("✓ Cinco outer folds disponíveis.")
print("✓ Arquiteturas selecionadas confirmadas.")

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path(
    "../data/processed/neural_sdm"
)

model_replicates = {}

for rep in range(1, 6):

    file_path = (
        data_dir /
        f"sapajus_neural_sdm_rep{rep}.csv"
    )

    model_replicates[rep] = pd.read_csv(
        file_path
    )

    print(
        f"Rep {rep}: "
        f"{len(model_replicates[rep])} registros"
    )

In [ ]:
predictors = [
    "BIO1",
    "BIO4",
    "BIO12",
    "BIO15"
]

print("Preditores:", predictors)

In [ ]:
print(
    "build_candidate_model disponível:",
    "build_candidate_model" in globals()
)

print(
    "candidate_models disponível:",
    "candidate_models" in globals()
)

In [ ]:
candidate_models = {
    "A_8": {
        "hidden_units": [8],
        "l2": 0.001
    },

    "B_16_8": {
        "hidden_units": [16, 8],
        "l2": 0.001
    },

    "C_16_8_L2": {
        "hidden_units": [16, 8],
        "l2": 0.01
    }
}

In [ ]:
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, regularizers


def build_candidate_model(
    hidden_units,
    l2_strength,
    seed=42
):

    tf.keras.utils.set_random_seed(seed)

    model = keras.Sequential()

    model.add(
        layers.Input(
            shape=(len(predictors),)
        )
    )

    for units in hidden_units:

        model.add(
            layers.Dense(
                units,
                activation="relu",
                kernel_regularizer=regularizers.l2(
                    l2_strength
                )
            )
        )

    model.add(
        layers.Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss="binary_crossentropy",

        metrics=[
            keras.metrics.AUC(
                name="roc_auc"
            ),

            keras.metrics.AUC(
                curve="PR",
                name="pr_auc"
            )
        ]
    )

    return model

In [ ]:
assert "build_candidate_model" in globals()
assert "candidate_models" in globals()
assert "model_replicates" in globals()
assert "predictors" in globals()
assert "selected_models" in globals()

assert set(
    selected_models["model"]
).issubset(candidate_models.keys())

print("✓ Arquiteturas recuperadas.")
print("✓ Função neural recuperada.")
print("✓ Réplicas disponíveis.")
print("✓ Seleção espacial disponível.")
print("✓ Ambiente pronto para treinamento.")

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import numpy as np
import pandas as pd
import tensorflow as tf

data = model_replicates[1].copy()

outer_results = []
outer_predictions = []

for _, selected in selected_models.iterrows():

    outer_fold = int(selected["outer_fold"])

    # Arredondamento convencional: 6.5 -> 7
    n_epochs = max(
        1,
        int(np.floor(
            float(selected["median_epoch"]) + 0.5
        ))
    )

    development = data[
        data["fold"] != outer_fold
    ].copy()

    external_test = data[
        data["fold"] == outer_fold
    ].copy()

    # Dados
    X_dev = development[predictors].to_numpy(
        dtype=np.float32
    )
    y_dev = development["class"].to_numpy(
        dtype=np.float32
    )

    X_test = external_test[predictors].to_numpy(
        dtype=np.float32
    )
    y_test = external_test["class"].to_numpy(
        dtype=np.float32
    )

    # Scaler ajustado somente no desenvolvimento
    scaler = StandardScaler()

    X_dev_scaled = scaler.fit_transform(
        X_dev
    ).astype(np.float32)

    X_test_scaled = scaler.transform(
        X_test
    ).astype(np.float32)

    # Pesos calculados somente no desenvolvimento
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=y_dev.astype(int)
    )

    class_weight = {
        0: float(weights[0]),
        1: float(weights[1])
    }

    # Modelo escolhido pela inner CV
    config = candidate_models[selected["model"]]

    tf.keras.backend.clear_session()

    model_outer = build_candidate_model(
        hidden_units=config["hidden_units"],
        l2_strength=config["l2"],
        seed=42
    )

    model_outer.fit(
        X_dev_scaled,
        y_dev,
        epochs=n_epochs,
        batch_size=32,
        class_weight=class_weight,
        verbose=0
    )

    # Predições externas
    probabilities = model_outer.predict(
        X_test_scaled,
        verbose=0
    ).ravel()

    roc_auc = roc_auc_score(
        y_test,
        probabilities
    )

    pr_auc = average_precision_score(
        y_test,
        probabilities
    )

    outer_results.append({
        "replicate": 1,
        "outer_fold": outer_fold,
        "model": selected["model"],
        "epochs": n_epochs,
        "n_train": len(development),
        "n_test": len(external_test),
        "n_test_presence": int(y_test.sum()),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "pr_baseline": float(y_test.mean())
    })

    fold_predictions = external_test.copy()

    fold_predictions["predicted_score"] = (
        probabilities
    )

    outer_predictions.append(fold_predictions)

    print(
        f"Fold {outer_fold} | "
        f"Epochs={n_epochs} | "
        f"ROC-AUC={roc_auc:.3f} | "
        f"PR-AUC={pr_auc:.3f}"
    )

In [ ]:
outer_results_df = pd.DataFrame(
    outer_results
)

outer_predictions_df = pd.concat(
    outer_predictions,
    ignore_index=True
)

assert len(outer_results_df) == 5

assert len(outer_predictions_df) == 1512

assert outer_predictions_df[
    "predicted_score"
].between(0, 1).all()

assert outer_predictions_df[
    "class"
].sum() == 252

display(
    outer_results_df.round(3)
)

In [ ]:
spatial_performance = pd.DataFrame({
    "metric": [
        "ROC-AUC",
        "PR-AUC"
    ],

    "mean": [
        outer_results_df["roc_auc"].mean(),
        outer_results_df["pr_auc"].mean()
    ],

    "sd": [
        outer_results_df["roc_auc"].std(),
        outer_results_df["pr_auc"].std()
    ],

    "min": [
        outer_results_df["roc_auc"].min(),
        outer_results_df["pr_auc"].min()
    ],

    "max": [
        outer_results_df["roc_auc"].max(),
        outer_results_df["pr_auc"].max()
    ]
})

display(spatial_performance.round(3))

In [ ]:
outer_results_df.to_csv(
    results_dir / "outer_results_rep1.csv",
    index=False
)

outer_predictions_df.to_csv(
    results_dir / "outer_predictions_rep1.csv",
    index=False
)

spatial_performance.to_csv(
    results_dir / "spatial_performance_rep1.csv",
    index=False
)

print("✓ Avaliação espacial externa da Rep 1 salva.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

data = model_replicates[1].copy()

fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 8)
)

for ax, variable in zip(
    axes.flat,
    predictors
):

    data.boxplot(
        column=variable,
        by="fold",
        ax=ax,
        grid=False
    )

    ax.set_title(variable)
    ax.set_xlabel("Spatial fold")
    ax.set_ylabel("Value")

plt.suptitle(
    "Environmental variation among spatial folds",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
environmental_extrapolation = []

data = model_replicates[1].copy()

for outer_fold in range(1, 6):

    train = data[
        data["fold"] != outer_fold
    ].copy()

    test = data[
        data["fold"] == outer_fold
    ].copy()

    train_min = train[predictors].min()
    train_max = train[predictors].max()

    outside = (
        (test[predictors] < train_min) |
        (test[predictors] > train_max)
    )

    environmental_extrapolation.append({
        "fold": outer_fold,
        "n_test": len(test),

        "pct_any_extrapolation":
            100 * outside.any(axis=1).mean(),

        **{
            f"pct_{variable}":
                100 * outside[variable].mean()
            for variable in predictors
        }
    })

extrapolation_df = pd.DataFrame(
    environmental_extrapolation
)

display(extrapolation_df.round(2))

In [ ]:
import matplotlib.pyplot as plt

data = model_replicates[1].copy()

fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9)
)

for ax, variable in zip(axes.flat, predictors):

    positions = []
    values = []
    colors = []

    for fold in range(1, 6):

        for class_value, offset, color in [
            (0, -0.18, "#B0B0B0"),
            (1, 0.18, "#2A788E")
        ]:

            subset = data[
                (data["fold"] == fold) &
                (data["class"] == class_value)
            ]

            values.append(
                subset[variable].dropna().values
            )

            positions.append(fold + offset)
            colors.append(color)

    bp = ax.boxplot(
        values,
        positions=positions,
        widths=0.30,
        patch_artist=True,
        manage_ticks=False,
        showfliers=False
    )

    for patch, color in zip(
        bp["boxes"], colors
    ):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax.set_xticks(range(1, 6))
    ax.set_xticklabels(range(1, 6))
    ax.set_xlabel("Spatial fold")
    ax.set_ylabel(variable)
    ax.set_title(variable)

from matplotlib.patches import Patch

fig.legend(
    handles=[
        Patch(
            facecolor="#B0B0B0",
            label="Pseudoabsence"
        ),
        Patch(
            facecolor="#2A788E",
            label="Presence"
        )
    ],
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 1.01)
)

fig.suptitle(
    "Environmental separation between presences and pseudoabsences",
    fontsize=14,
    y=1.06
)

plt.tight_layout()
plt.show()

In [ ]:
print(model_replicates[1].columns.tolist())

In [ ]:
import pandas as pd

coord_cols = ["x_5880", "y_5880"]

reference = (
    model_replicates[1]
    .loc[
        model_replicates[1]["class"] == 1,
        coord_cols + ["fold"]
    ]
    .sort_values(coord_cols)
    .reset_index(drop=True)
)

print("Presenças de referência:", len(reference))

for rep in range(2, 6):

    current = (
        model_replicates[rep]
        .loc[
            model_replicates[rep]["class"] == 1,
            coord_cols + ["fold"]
        ]
        .sort_values(coord_cols)
        .reset_index(drop=True)
    )

    identical = reference.equals(current)

    print(
        f"Rep {rep}: "
        f"{len(current)} presenças | "
        f"coordenadas e folds idênticos = {identical}"
    )

In [ ]:
import pandas as pd

audit = []

for rep in range(1, 6):

    df = model_replicates[rep]

    pa = df.loc[
        df["class"] == 0
    ]

    audit.append({
        "replicate": rep,
        "n_total": len(df),
        "n_presence": int((df["class"] == 1).sum()),
        "n_pseudoabsence": len(pa),
        "n_folds": df["fold"].nunique(),
        "n_unique_pa": len(
            pa[["x_5880", "y_5880"]].drop_duplicates()
        )
    })

pa_audit = pd.DataFrame(audit)

display(pa_audit)

assert (pa_audit["n_presence"] == 252).all()
assert (pa_audit["n_pseudoabsence"] == 1260).all()
assert (pa_audit["n_folds"] == 5).all()
assert (pa_audit["n_unique_pa"] == 1260).all()

print("✓ Estrutura das cinco réplicas validada.")

In [ ]:
pa_sets = {}

for rep in range(1, 6):

    pa = model_replicates[rep].loc[
        model_replicates[rep]["class"] == 0
    ]

    pa_sets[rep] = set(
        zip(pa["x_5880"], pa["y_5880"])
    )

for rep in range(2, 6):

    overlap = len(
        pa_sets[1].intersection(pa_sets[rep])
    )

    print(
        f"Rep 1 × Rep {rep}: "
        f"{overlap} pseudoausências compartilhadas "
        f"({100 * overlap / 1260:.2f}%)"
    )

In [ ]:
print(
    "evaluate_candidate disponível:",
    "evaluate_candidate" in globals()
)

print(
    "build_candidate_model disponível:",
    "build_candidate_model" in globals()
)

print(
    "candidate_models disponível:",
    "candidate_models" in globals()
)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

import numpy as np
import tensorflow as tf
from tensorflow import keras


def evaluate_candidate(
    data,
    predictors,
    outer_test_fold,
    inner_val_fold,
    hidden_units,
    l2_strength,
    seed=42
):

    # Separação espacial
    train = data[
        ~data["fold"].isin(
            [outer_test_fold, inner_val_fold]
        )
    ].copy()

    val = data[
        data["fold"] == inner_val_fold
    ].copy()

    # Preditores e classes
    X_train = train[predictors].to_numpy(
        dtype=np.float32
    )

    y_train = train["class"].to_numpy(
        dtype=np.float32
    )

    X_val = val[predictors].to_numpy(
        dtype=np.float32
    )

    y_val = val["class"].to_numpy(
        dtype=np.float32
    )

    # Padronização somente no treino
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    ).astype(np.float32)

    X_val_scaled = scaler.transform(
        X_val
    ).astype(np.float32)

    # Pesos das classes somente no treino
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=y_train.astype(int)
    )

    class_weight = {
        0: float(weights[0]),
        1: float(weights[1])
    }

    # Nova rede neural
    tf.keras.backend.clear_session()

    model = build_candidate_model(
        hidden_units=hidden_units,
        l2_strength=l2_strength,
        seed=seed
    )

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=30,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_scaled,
        y_train,

        validation_data=(
            X_val_scaled,
            y_val
        ),

        epochs=300,
        batch_size=32,

        class_weight=class_weight,
        callbacks=[early_stopping],

        verbose=0
    )

    # Predições na validação interna
    probabilities = model.predict(
        X_val_scaled,
        verbose=0
    ).ravel()

    best_epoch = (
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    return {
        "roc_auc": roc_auc_score(
            y_val,
            probabilities
        ),

        "pr_auc": average_precision_score(
            y_val,
            probabilities
        ),

        "val_loss": min(
            history.history["val_loss"]
        ),

        "best_epoch": int(best_epoch)
    }

In [ ]:
assert callable(evaluate_candidate)
assert callable(build_candidate_model)

assert len(candidate_models) == 3
assert len(model_replicates) == 5

assert set(model_replicates.keys()) == {
    1, 2, 3, 4, 5
}

print("✓ Função de validação interna recuperada.")
print("✓ Três arquiteturas disponíveis.")
print("✓ Cinco réplicas disponíveis.")
print("✓ Ambiente pronto para nested spatial CV.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

nested_dir = (
    Path("../data/processed/neural_sdm")
    / "nested_cv"
)

nested_dir.mkdir(
    parents=True,
    exist_ok=True
)

for rep in range(2, 6):

    print(f"\n{'=' * 45}")
    print(f"REPLICATE {rep}")
    print(f"{'=' * 45}")

    data = model_replicates[rep].copy()

    for outer_fold in range(1, 6):

        checkpoint = (
            nested_dir /
            f"rep{rep}_outer{outer_fold}_selection.csv"
        )

        # Não repetir avaliações concluídas
        if checkpoint.exists():

            saved = pd.read_csv(checkpoint)

            expected = {
                (name, fold)
                for name in candidate_models
                for fold in range(1, 6)
                if fold != outer_fold
            }

            observed = set(
                zip(
                    saved["model"],
                    saved["inner_val_fold"]
                )
            )

            if (
                len(saved) == 12
                and observed == expected
                and saved["roc_auc"].notna().all()
                and saved["best_epoch"].ge(1).all()
            ):
                print(
                    f"Rep {rep} | Outer {outer_fold}: "
                    "checkpoint válido, ignorando."
                )
                continue

            raise ValueError(
                f"Checkpoint incompleto: {checkpoint}"
            )

        print(
            f"\nRep {rep} | Outer fold {outer_fold}"
        )

        inner_folds = [
            f for f in range(1, 6)
            if f != outer_fold
        ]

        fold_results = []

        for model_name, config in candidate_models.items():

            for inner_val_fold in inner_folds:

                result = evaluate_candidate(
                    data=data,
                    predictors=predictors,
                    outer_test_fold=outer_fold,
                    inner_val_fold=inner_val_fold,
                    hidden_units=config["hidden_units"],
                    l2_strength=config["l2"],
                    seed=42
                )

                fold_results.append({
                    "replicate": rep,
                    "outer_fold": outer_fold,
                    "model": model_name,
                    "inner_val_fold": inner_val_fold,
                    **result
                })

                print(
                    f"  {model_name} | "
                    f"Inner {inner_val_fold} | "
                    f"ROC={result['roc_auc']:.3f}"
                )

        fold_df = pd.DataFrame(fold_results)

        # Salvar somente após concluir as 12 avaliações
        fold_df.to_csv(
            checkpoint,
            index=False
        )

        print(
            f"✓ Checkpoint salvo: "
            f"Rep {rep}, Outer {outer_fold}"
        )

print("\n✓ Seleção interna das réplicas 2–5 concluída.")